# 01. Memory management

**Tier:** Deep. **Why you'd use this:** Objects free when nothing references them, but a cycle can survive plain `del`. Prefer `with` or `finally` over `__del__` for cleanup.

**After this lesson you can:** say when a reference count changes; explain why a cycle survives `del`; choose `with` or `finally` instead of `__del__`.
About 30 minutes. **Uses:** [Objects, names, references](../01-objects/01-names-lesson.ipynb), [Mutability and aliasing](../02-mutability/01-aliasing-lesson.ipynb)


## Quick review

1. What does `compile` return?
2. Why are locals cheaper to load than globals in bytecode?

<details><summary>Answers</summary>

A code object (then a function wraps it). Locals use a fast index; globals need a name lookup.
</details>


## The idea

CPython keeps a count of how many references point at an object. The count drops when a name is rebound, a container drops the object, or a frame ends. At zero the object is freed immediately. That is reference counting.

A cycle is two or more objects that point at each other and at nothing else you still hold. Each count stays at least 1, so reference counting never reaches zero. A separate collector, `gc`, finds those cycles and frees them. It does not run on every `del`.

<details><summary>Explain more</summary>

`sys.getrefcount` reports the count and is a CPython detail. The call itself holds a temporary reference, so the number is higher than the names you can see. Compare two calls. The difference is the references you added.

`weakref.ref` does not count. When the object is freed, the weak reference's call returns `None`. That is how the lesson watches an object die without keeping it alive.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

Those runtimes are tracing collectors first. CPython is reference counting plus a cycle collector. `del` drops a reference. It is not a destructor you can schedule, and `__del__` is not a `finally`.
</details>


## See it work

A handle must close even when the work raises, and even if some other name still points at the handle. `__del__` waits until the count hits zero, which is the wrong time: you still have the name, and a cycle may keep the count above zero. `finally` runs when the block ends.


In [1]:
import sys


class Handle:
    def __init__(self) -> None:
        self.closed = False

    def close(self) -> None:
        self.closed = True


def use(handle: Handle) -> str:
    # 1. Close on the way out, success or exception.
    try:
        raise ValueError("boom")
    finally:
        handle.close()


handle = Handle()
try:
    use(handle)
except ValueError:
    print(handle.closed)
print(sys.getrefcount(handle) > 1)


True
True


| moment | `handle.closed` | why |
| --- | --- | --- |
| inside `try`, before the raise | False | not closed yet |
| `finally` | True | the block is ending |
| after the exception is caught | True | the name `handle` still exists |

```
try: raise
finally: close   # runs before the except outside sees the error
name handle -----> Handle(closed=True)
```

`__del__` would not have run in that `finally` line. The count is still above zero because `handle` points at the object.


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `getrefcount` not equal the number of names you can see?
2. Why does `del` on a cycle leave the weak reference alive?
3. Why can `handle.closed` be read after `finally` closes it?

<details><summary>Answers</summary>

1. The argument to `getrefcount` is a reference for the duration of the call. Containers and frames you are not thinking about count too. Subtract two readings instead of trusting one integer.
2. Each node still references the other. Reference counting only frees at zero. The collector is what notices that the group is unreachable from your names.
3. `finally` does not require the count to be zero. It calls `close` while your name still points at the handle. That is the point of the block.
</details>


In [2]:
import sys

left = []
before = sys.getrefcount(left)
right = left
after = sys.getrefcount(left)
print(after - before)


1


<details><summary>Check your answers</summary>

- The difference is `1`. `right = left` added one real reference. Both calls to `getrefcount` hold their own temporary reference, and that temporary is in both numbers, so it cancels in the subtraction.
</details>


## Why it works this way

Every pointer we have been drawing is a reference the count includes: a name in a frame, an entry in a list or dict, a class attribute, a bound method's `__self__`. Creating the pointer increments. Dropping it decrements. Reaching zero frees the object at once, which is why an ordinary temporary disappears without a collector pause. The cost is the bookkeeping on every store and every drop. The design reason is prompt release of scarce things when the last reference goes, without waiting for a scan of the whole heap.

Cycles defeat that promptness. If `left.other` is `right` and `right.other` is `left`, deleting your names drops two references and leaves each object with a count of 1, held by the other. The memory stays. `gc.collect` walks containers the collector knows about, finds groups whose only remaining references are inside the group, and frees them. It also runs on its own from time to time. It does not run because you wrote `del`. A weak reference does not add to the count, so it can observe the death: before the collect the call returns the object, and after it returns `None`.

`__del__` is a method the interpreter may call when the count hits zero. It is the wrong place for a lock, a file, or a socket. The call can be delayed by a cycle. The call can be skipped if the process is shutting down and the objects `__del__` needs are already gone. The call can resurrect the object by storing `self` somewhere, and then the count is positive again. Order among several `__del__` methods is not a promise. `finally` and `with` do not wait for any of that. They run at the end of the block, on the way out, including when the block raises. The handle in the worked example is still referenced after `close`, which is what you want: you can still read `closed`. A destructor that required the object to be unreachable could not support that.

`sys.getrefcount` is a CPython detail. Other Python implementations may not use reference counting at all, and the integer you see includes the temporary in the call. Subtract two readings taken the same way. Do not write a program that needs the count to be a particular number. The stable fact in the predict cell is the difference of 1.

The cycle cell is the other fact. After `del` of both names, the weak references still return the nodes. After `gc.collect`, they return `None`. An object that is not in a cycle dies at the `del`, with no collect, which is the third cell.


In [3]:
import gc
import weakref


class Node:
    def __init__(self) -> None:
        self.other = None


left = Node()
right = Node()
left.other = right
right.other = left
watch_left = weakref.ref(left)
del left
del right
print(watch_left() is None)
gc.collect()
print(watch_left() is None)


False
True


In [4]:
class Lone:
    pass


lone = Lone()
watch = weakref.ref(lone)
print(watch() is lone)
del lone
print(watch() is None)


True
True


## When to use it

Use `with` or `finally` for anything that must be released: files, locks, sockets, database transactions. Use `del` only to drop a reference you no longer want, knowing that a cycle or another name can keep the object. Use `weakref` for a cache that must not keep objects alive, such as a map from objects to extra data that should vanish with the object.

Do not write `__del__` to close resources. If you need a last-chance log while exploring, remember it can run late or not at all. Do not call `gc.collect` in ordinary code to "free memory". Break the cycle, or let the collector run. Call it in a test or a lesson when you need to show that a cycle was the thing keeping the objects.

A long-lived process that builds cycles of its own objects will grow until the collector runs. Prefer not to build the cycle: a parent points at children, and children do not point back, or the back pointer is weak.


## Common mistake

Deleting the names of a cycle and assuming the objects are gone. The weak reference still returns a node. Nothing raises. A later `gc.collect` is what drops the counts the cycle was holding up.


In [5]:
left = Node()
right = Node()
left.other = right
right.other = left
watch = weakref.ref(left)
del left
del right
print(watch() is None)


False


The print is `False`. The two nodes point at each other, so each count is still at least 1. The fix breaks the cycle before dropping the names. After `left.other = None`, the nodes are an ordinary chain, and deleting the names frees them without waiting for the collector.


In [6]:
left = Node()
right = Node()
left.other = right
right.other = left
watch = weakref.ref(left)
left.other = None
right.other = None
del left
del right
print(watch() is None)


True


## Try it

Add a third node so `left.other` is `right`, `right.other` is `extra`, and `extra.other` is `left`. After deleting all three names, is the weak reference dead before `gc.collect`? What one assignment breaks the cycle without deleting any node?


## What this explains

- [Objects, names, references](../01-objects/01-names-lesson.ipynb): a name is a reference. The count is how many of those pointers exist.
- [Mutability and aliasing](../02-mutability/01-aliasing-lesson.ipynb): two names on one list are two references. A cycle is the case where the pointers form a loop.
- [Context managers](../../course/12-data-model/02-context-managers-lesson.ipynb): `with` runs the cleanup at the end of the block. That is `finally` with a protocol, and it does not wait for `__del__`.


## Recap

- CPython frees an object when its reference count hits zero. `del` drops a reference. It does not schedule a destructor.
- A cycle keeps counts above zero. `gc.collect` reclaims the group. A weak reference does not keep the object alive.
- Close resources in `finally` or `with`. `__del__` can run late, never, or in an undefined order.

Next: [The GIL and free-threading](../12-gil/01-gil-lesson.ipynb).
